# CS4603 PA4 — Document Analyst

Development & testing notebook. Section headers match the tasks in `README.md`.
Fill in each cell, run everything top-to-bottom, and **keep all outputs visible** before submitting.
Record explanations and analysis answers in `STUDENT_ANALYSIS.md`.


## Part 0 — Setup & Corpus Ingestion
Env config + ingest `data/annual_report.pdf` into Databricks Vector Search (Task 0.3).


In [4]:
# TODO(0.1): load config / verify env vars
# from config import ...

from config import get_settings

settings = get_settings()

print("Host:", settings["host"])
print("Model:", settings["model"])
print("Embeddings:", settings["embeddings"])
print("Vector Search Endpoint:", settings["vs_endpoint"])
print("Vector Search Index:", settings["vs_index"])

Host: https://dbc-be287b92-99e7.cloud.databricks.com
Model: databricks-meta-llama-3-3-70b-instruct
Embeddings: databricks-gte-large-en
Vector Search Endpoint: analyst-vs-endpoint
Vector Search Index: cs4603.default.analyst_index


In [5]:
from databricks.connect import DatabricksSession
spark = (
    DatabricksSession.builder
    .remote(host=settings["host"], token=settings["token"])
    .serverless(True)
    .getOrCreate()
)

In [6]:
# TODO(0.3): ingest corpus -> Delta table -> Vector Search index; wait until READY
# from rag.ingest import ingest
# ingest(spark, volume_path='/Volumes/main/default/pa4/annual_report.pdf')
from rag.ingest import build_chunks_table

build_chunks_table(spark, "/Volumes/main/default/pa4/annual_report.pdf", "main.default.analyst_chunks")


{"ts": "2026-07-18 04:38:40.423", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVABLE_TABLE_VALUED_FUNCTION] Could not resolve ai_prep_search to a table-valued function.\nPlease make sure that ai_prep_search is defined as a table-valued function and that all required parameters are provided correctly.\nIf ai_prep_search is not defined, please create the table-valued function before using it.\nFor more information about defining table-valued functions, please refer to the Apache Spark documentation. SQLSTATE: 42883; line 4 pos 9;\n'CreateTableAsSelect TableSpec(Map(),None,Map(),None,None,None,None,false,false,Set(),None,None,None,None,List()), false, false\n:- ResolvedIdentifier com.databricks.sql.managedcatalog.UnityCatalogV2Proxy@1ac299bc, default.analyst_chunks\n+- 'Project [*]\n   +- 'UnresolvedTableValuedFunction [ai_prep_search], [AIRemoteFunction(/Volumes/main/default/pa4/annual_report.pdf, Map(), ai_parse_document, VariantType)], false\n\n\nJVM stacktrace:

AnalysisException: [UNRESOLVABLE_TABLE_VALUED_FUNCTION] Could not resolve ai_prep_search to a table-valued function.
Please make sure that ai_prep_search is defined as a table-valued function and that all required parameters are provided correctly.
If ai_prep_search is not defined, please create the table-valued function before using it.
For more information about defining table-valued functions, please refer to the Apache Spark documentation. SQLSTATE: 42883; line 4 pos 9;
'CreateTableAsSelect TableSpec(Map(),None,Map(),None,None,None,None,false,false,Set(),None,None,None,None,List()), false, false
:- ResolvedIdentifier com.databricks.sql.managedcatalog.UnityCatalogV2Proxy@1ac299bc, default.analyst_chunks
+- 'Project [*]
   +- 'UnresolvedTableValuedFunction [ai_prep_search], [AIRemoteFunction(/Volumes/main/default/pa4/annual_report.pdf, Map(), ai_parse_document, VariantType)], false


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.cata

## Part 1 — Build the Document Analyst graph
Nodes: planner (1.2), supervisor (1.3), RAG agent (1.4), MCP tools (1.5), synthesizer (1.6), full graph (1.7).


In [ ]:
# TODO(1.7): build the compiled graph
# from agent.graph import build_graph
# graph = build_graph()



In [ ]:
# TODO(1.7): visualize the compiled graph
# from IPython.display import Image
# Image(graph.get_graph().draw_mermaid_png())



### Test the graph


In [ ]:
# Retrieval-only query
# graph.invoke({'messages':[{'role':'user','content':'What was the net income in 2023?'}]})



In [ ]:
# Computation-only query
# graph.invoke({'messages':[{'role':'user','content':'What is 15% of 2.4 billion?'}]})



In [ ]:
# Combined query — show the full step-by-step execution trace
# graph.invoke({'messages':[{'role':'user','content':'What was the revenue in 2023, and what would a 10% increase look like?'}]})



### Required — offline smoke test
Runs the graph with a mocked LLM (no Databricks calls). Same test Bonus A automates.


In [ ]:
!python -m pytest tests/test_smoke.py -q


## Part 2 — Deployment
Package as an MLflow models-from-code model, register in Unity Catalog, create the serving endpoint (Tasks 2.1–2.4).
Reference: `databricks_deployment_v1/deployment.ipynb`.


In [ ]:
# TODO(2.1): sanity-check the model definition imports cleanly
!python -c "import deployment.agent_model"



In [ ]:
# TODO(2.2): log + register the model version in Unity Catalog



In [ ]:
# TODO(2.3): create/update the serving endpoint; wait for READY; print the URL



### Test the deployed endpoint (Task 2.4)


In [ ]:
# curl the endpoint and show the raw response



In [ ]:
# Response shape depends on how you logged the model (see README Task 2.4 / GUIDE §7).
#
# Path A — raw LangGraph state (mlflow.langchain.log_model, v1 style):
# import requests
# url = f'{DATABRICKS_HOST}/serving-endpoints/<your-endpoint-name>/invocations'
# r = requests.post(url, headers={'Authorization': f'Bearer {DATABRICKS_TOKEN}'},
#                   json={'messages':[{'role':'user','content':'What was the net income in 2023?'}]})
# print(r.json()[0]['messages'][-1]['content'])
#
# Path B — OpenAI ChatCompletion (mlflow.pyfunc.ChatModel / ChatAgent, v2 style):
# import openai
# client = openai.OpenAI(api_key=DATABRICKS_TOKEN, base_url=f'{DATABRICKS_HOST}/serving-endpoints')
# resp = client.chat.completions.create(model='<your-endpoint-name>',
#     messages=[{'role':'user','content':'What was the net income in 2023?'}])
# print(resp.choices[0].message.content)



## Part 3 — Client SDK demo
Instantiate `DocumentAnalystClient`, health-check, ask, stream, and show timeout/retry handling (Task 3.2).


In [ ]:
# from client.sdk import DocumentAnalystClient
# c = DocumentAnalystClient(...)
# assert c.health_check() is True
# print(c.ask('What was the net income in 2023?'))



In [ ]:
# ask_streaming demo
# for chunk in c.ask_streaming('Summarize FY2023 revenue.'): print(chunk, end='')



In [ ]:
# Simulate timeout (timeout=0.001) and endpoint-unavailable retry behavior

